# 02 Train/Evaluate ModernVBERT Retrieval

This notebook consumes a fixed OCR artifact. It does not install or import PaddleOCR, OpenCV, EasyOCR, or PaddlePaddle.

## 1. Retrieval Environment

Run once, restart the runtime, then continue.

In [ ]:
!pip -q uninstall -y paddleocr paddlex paddlepaddle paddlepaddle-gpu easyocr opencv-python opencv-contrib-python opencv-python-headless
!pip -q uninstall -y pillow
!pip -q install "pillow==11.3.0" "PyYAML==6.0.2"
!pip -q install -U --upgrade-strategy only-if-needed \
    "datasets==2.21.0" \
    git+https://github.com/huggingface/transformers.git \
    "colpali-engine==0.3.15" \
    "accelerate==0.34.2" \
    "sentencepiece" "sacremoses" "tqdm"

# Optional LoRA appendix only:
# !pip -q install "peft==0.18.0"


## 2. Load Retrieval Module

In [ ]:
import sys
from pathlib import Path

MODULE_DIR = Path('/kaggle/input/datasets/kysyandrewik/rumodernvbert')
if not MODULE_DIR.exists():
    MODULE_DIR = Path.cwd()
sys.path.insert(0, str(MODULE_DIR))

import modernvbert_ru_retrieval_v2 as pipeline_v2

WORKDIR = pipeline_v2.default_workdir('modernvbert_ru_retrieval_v2')
print('WORKDIR:', WORKDIR)
print('module:', pipeline_v2.__file__)


## 3. Configuration

In [ ]:
RETRIEVAL_MODEL_NAME = 'ModernVBERT/colmodernvbert-merged'
EVAL_DEVICE = 'cuda'

# Point this to the Kaggle Dataset root created by 01_prepare_synthetic_dataset_paddleocr.ipynb.
PREPARED_DIR = Path('/kaggle/input/modernvbert-ru-ocr-artifact')

QUERY_ADAPTER_CONFIG = {
    'mode': 'query_adapter',
    'num_epochs': 1,
    'batch_size': 4,
    'learning_rate': 1e-3,
    'weight_decay': 0.01,
    'hidden_dim': 512,
    'dropout': 0.05,
    'residual_scale': 0.1,
    'max_train_rows': 256,
    'one_query_per_doc': True,
    'image_resolution': None,
    'attention_mode': 'trim',
    'seed': 42,
}

RUN_QUERY_ADAPTER = True
RUN_SYNTHETIC_QUERY_ADAPTER = True
RUN_DUAL_ADAPTER = False
RUN_ROBUSTNESS = True
RUN_MWS_VISION = True
MWS_MAX_ROWS = 200


## 4. Preflight And Load Artifact

In [ ]:
preflight = pipeline_v2.assert_v2_environment_ready(
    require_ocr=False,
    require_retriever=True,
    model_name=RETRIEVAL_MODEL_NAME,
    device=EVAL_DEVICE,
)
pipeline_v2.export_json(WORKDIR / 'artifacts' / 'v2_preflight.json', preflight)

summary = pipeline_v2.load_prepared_experiment_artifacts(
    prepared_dir=PREPARED_DIR,
    work_dir=WORKDIR,
    validate=True,
)
pipeline_v2.export_json(WORKDIR / 'artifacts' / 'prepared_artifact_validation.json', summary['validation'])
summary['validation']


## 5. Zero-Shot Baselines

In [ ]:
zero_shot_results = []
for run_name, manifest_path in [
    ('english_reference_zero_shot', summary['manifests']['english_reference']['test']),
    ('ru_query_original_image_zero_shot', summary['manifests']['russian_baseline']['test']),
    ('ru_query_synthetic_image_zero_shot', summary['manifests']['synthetic_primary']['test']),
]:
    result = pipeline_v2.evaluate_manifest_v2(
        manifest_path=manifest_path,
        model_name=RETRIEVAL_MODEL_NAME,
        run_name=run_name,
        output_path=WORKDIR / 'results_v2' / f'{run_name}.json',
        device=EVAL_DEVICE,
    )
    zero_shot_results.append({'run_name': run_name, **result['metrics'], **result['efficiency']})
zero_shot_results


## 6. Early-Fusion Text Proxy

In [ ]:
early_fusion_proxy = pipeline_v2.evaluate_early_fusion_text_proxy(
    manifest_path=summary['manifests']['russian_baseline']['test'],
    output_dir=WORKDIR / 'results_v2' / 'early_fusion_proxy',
    run_name='early_fusion_text_proxy_bm25',
)
early_fusion_proxy['metrics']


## 7. Train Query Adapters

In [ ]:
baseline_adapter = None
synthetic_adapter = None

if RUN_QUERY_ADAPTER:
    baseline_adapter = pipeline_v2.train_embedding_adapter(
        train_manifest=summary['manifests']['russian_baseline']['train'],
        val_manifest=summary['manifests']['russian_baseline']['val'],
        output_dir=WORKDIR / 'adapters_v2' / 'ru_query_original_image_query_adapter',
        model_name=RETRIEVAL_MODEL_NAME,
        device=EVAL_DEVICE,
        **QUERY_ADAPTER_CONFIG,
    )

if RUN_SYNTHETIC_QUERY_ADAPTER:
    synthetic_adapter = pipeline_v2.train_embedding_adapter(
        train_manifest=summary['manifests']['synthetic_primary']['train'],
        val_manifest=summary['manifests']['synthetic_primary']['val'],
        output_dir=WORKDIR / 'adapters_v2' / 'ru_query_synthetic_image_query_adapter',
        model_name=RETRIEVAL_MODEL_NAME,
        device=EVAL_DEVICE,
        **QUERY_ADAPTER_CONFIG,
    )

{'baseline_adapter': baseline_adapter, 'synthetic_adapter': synthetic_adapter}


## 8. Optional Dual Adapter

In [ ]:
dual_adapter = None
if RUN_DUAL_ADAPTER:
    dual_config = dict(QUERY_ADAPTER_CONFIG)
    dual_config['mode'] = 'dual_adapter'
    dual_config['max_train_rows'] = min(128, dual_config['max_train_rows'])
    dual_adapter = pipeline_v2.train_embedding_adapter(
        train_manifest=summary['manifests']['synthetic_primary']['train'],
        val_manifest=summary['manifests']['synthetic_primary']['val'],
        output_dir=WORKDIR / 'adapters_v2' / 'synthetic_dual_adapter',
        model_name=RETRIEVAL_MODEL_NAME,
        device=EVAL_DEVICE,
        **dual_config,
    )
dual_adapter


## 9. Main Suite

In [ ]:
main_results = pipeline_v2.evaluate_main_v2_suite(
    manifest_groups=summary['manifests'],
    output_dir=WORKDIR / 'results_v2' / 'main_suite',
    model_name=RETRIEVAL_MODEL_NAME,
    query_adapter_dir=baseline_adapter['best_query_adapter_dir'] if baseline_adapter else None,
    synthetic_query_adapter_dir=synthetic_adapter['best_query_adapter_dir'] if synthetic_adapter else None,
    dual_query_adapter_dir=dual_adapter['best_query_adapter_dir'] if dual_adapter else None,
    dual_image_adapter_dir=dual_adapter['best_image_adapter_dir'] if dual_adapter else None,
    aligned_test_manifests=summary['aligned_test_manifests'],
    device=EVAL_DEVICE,
)
main_results


## 10. Robustness

In [ ]:
robustness_results = None
if RUN_ROBUSTNESS:
    robustness_results = pipeline_v2.evaluate_robustness_suite(
        clean_manifest_path=summary['manifests']['russian_baseline']['test'],
        output_dir=WORKDIR / 'results_v2' / 'robustness',
        model_name=RETRIEVAL_MODEL_NAME,
        query_adapter_dir=baseline_adapter['best_query_adapter_dir'] if baseline_adapter else None,
        device=EVAL_DEVICE,
    )
robustness_results


## 11. MWS Vision Bench

In [ ]:
mws_results = None
if RUN_MWS_VISION:
    mws_summary = pipeline_v2.export_mws_vision_retrieval_manifest(
        output_dir=WORKDIR / 'mws_vision',
        max_rows=MWS_MAX_ROWS,
    )
    mws_zero = pipeline_v2.evaluate_manifest_v2(
        manifest_path=mws_summary['manifest_path'],
        model_name=RETRIEVAL_MODEL_NAME,
        run_name='mws_vision_zero_shot',
        output_path=WORKDIR / 'results_v2' / 'mws_vision_zero_shot.json',
        device=EVAL_DEVICE,
    )
    mws_adapted = pipeline_v2.evaluate_manifest_v2(
        manifest_path=mws_summary['manifest_path'],
        model_name=RETRIEVAL_MODEL_NAME,
        query_adapter_dir=baseline_adapter['best_query_adapter_dir'] if baseline_adapter else None,
        run_name='mws_vision_query_adapter',
        output_path=WORKDIR / 'results_v2' / 'mws_vision_query_adapter.json',
        device=EVAL_DEVICE,
    )
    mws_by_type = pipeline_v2.evaluate_by_metadata_group(
        manifest_path=mws_summary['manifest_path'],
        group_key='mws_type',
        output_dir=WORKDIR / 'results_v2' / 'mws_by_type',
        model_name=RETRIEVAL_MODEL_NAME,
        query_adapter_dir=baseline_adapter['best_query_adapter_dir'] if baseline_adapter else None,
        device=EVAL_DEVICE,
    )
    mws_results = {'summary': mws_summary, 'zero': mws_zero['metrics'], 'adapted': mws_adapted['metrics'], 'by_type': mws_by_type}
mws_results


## 12. Thesis Tables

In [ ]:
all_rows = list(main_results)
all_rows.append({'run_name': early_fusion_proxy['run_name'], **early_fusion_proxy['metrics'], **early_fusion_proxy['efficiency']})
if mws_results is not None:
    all_rows.extend(mws_results['by_type'])
if robustness_results is not None:
    all_rows.extend(robustness_results)

tables = pipeline_v2.export_thesis_tables_v2(all_rows, WORKDIR / 'thesis_outputs_v2')
tables
